# Negative coefficient preprocessing for QUBO solving

This tutorial shows how to handle negative off-diagonal coefficients using bit-flip preprocessing and zeroing.
Bit-flip preprocessing keeps the problem equivalent up to a variable transformation.
Zeroing is different: it changes the QUBO. It's automatically performed (and logged) by the quantum solver when preprocessing is not enough, since the solver only handles non-negative off-diagonal coefficients. When using the functional API, zeroing has to be applied explicitly. Embedders, on the other hand, raise an error if any negative off-diagonal coefficient is detected.

In [ ]:
import json
import math

import qoolqit

from qubosolver import (
    Analyzer,
    Instance,
    Solution,
    Solver,
    LocalEmulator,
    SolverConfig,
    solvers,
    transforms,
    drive_shaping,
    embedding,
    matrix,
)

/home/bussy/work/qubo-solver.git/negative-coefficients-bitflip-preprocessing/.venv/lib/python3.10/site-packages/qoolqit/execution/backends.py:10: DeprecationWarning: This package is replaced by pasqal-cloud (https://pypi.org/project/pasqal-cloud/) and is going to be removed in the future.
  from pulser_pasqal.backends import EmuFreeBackendV2, RemoteEmulatorBackend
/home/bussy/work/qubo-solver.git/negative-coefficients-bitflip-preprocessing/qubosolver/drive_shaping/_waveforms.py:8: DeprecationWarning: Constant is deprecated and will be removed in v1.4. Use the equivalent ConstantWaveform instead.
  from qoolqit import Constant as ConstantWaveform
/home/bussy/work/qubo-solver.git/negative-coefficients-bitflip-preprocessing/qubosolver/drive_shaping/_waveforms.py:8: DeprecationWarning: Constant is deprecated and will be removed in v1.4. Use the equivalent ConstantWaveform instead.
  from qoolqit import Constant as ConstantWaveform


In [ ]:
def print_solution(name, solution):
    print(name)
    df = Analyzer(solution).df
    df["Trivial"] = df["bitstrings"].apply(lambda b: len(set(b)) == 1)
    print(df)

## 1. A QUBO with negative off-diagonal coefficients

We start with a small QUBO that contains negative off-diagonal coefficients.

The example is chosen so that:

- the solution is not trivial;
- bit-flip preprocessing can remove all negative off-diagonal coefficients.

In [ ]:
instance = Instance(
    matrix.tensor(
    [
        [1.0, -3.0, -2.0, 1.0],
        [-3.0, 6.0, 3.0, -1.0],
        [-2.0, 3.0, 2.0, -1.0],
        [1.0, -1.0, -1.0, -1.0],
    ])
)

print(f"QUBO:\n{instance.matrix}")

QUBO:
tensor([[ 1., -3., -2.,  1.],
        [-3.,  6.,  3., -1.],
        [-2.,  3.,  2., -1.],
        [ 1., -1., -1., -1.]])


## 2. Bit-flip preprocessing

The bit-flip preprocessing chooses which binary variables should be complemented.

For each variable:

- if `flip_vector[i] = 0`, the variable is kept unchanged;
- if `flip_vector[i] = 1`, the variable is replaced by its complement.

Internally, the flip vector is selected by solving a variant of a max-cut problem with GLPK:
flipping a variable inverts the sign of every coefficient linking it to an unflipped variable, so
choosing the flip set is equivalent to choosing which "side" of a cut each variable falls on.

The objective is to minimize the total weight of negative off-diagonal coefficients remaining after the flips.

The transform lives in `qubosolver.transforms.negative_bitflip`. Calling `negative_bitflip.apply` solves the ILP, applies the flips, and returns an `Instance` that records the flip vector, status, and metrics.

In [ ]:
flipped_instance = transforms.negative_bitflip.apply(instance, time_limit_s=60.0)

print(f"Flip vector: {flipped_instance.flips}")
print(f"Bitflip status: {flipped_instance.status}")
print(f"Bitflip metrics: {json.dumps(flipped_instance.metrics, indent=4)}")

print(f"\nPreprocessed QUBO:\n{flipped_instance.matrix}")

Flip vector: tensor([1, 0, 0, 1], dtype=torch.int8)
Bitflip status: OPTIMAL
Bitflip metrics: {
    "n_edges": 6,
    "neg_count_before": 4,
    "neg_count_after": 0,
    "neg_count_reduction_pct": 100.0,
    "neg_weight_before": 7.0,
    "neg_weight_after": 0.0,
    "neg_weight_reduction_pct": 100.0,
    "objective_value": -7.0
}

Preprocessed QUBO:
tensor([[-3.,  3.,  2.,  1.],
        [ 3., -2.,  3.,  1.],
        [ 2.,  3., -4.,  1.],
        [ 1.,  1.,  1., -1.]])


In [ ]:
flip_vector = flipped_instance.flips

print(f"Flip vector: {flip_vector}")

for i, flip in enumerate(flip_vector.tolist()):
    if flip == 0:
        print(f"x_{i} = y_{i}")
    else:
        print(f"x_{i} = 1 - y_{i}")

Flip vector: tensor([1, 0, 0, 1], dtype=torch.int8)
x_0 = 1 - y_0
x_1 = y_1
x_2 = y_2
x_3 = 1 - y_3


## 3. Classical sanity check with brute force

We solve the same QUBO exhaustively:

1. without preprocessing;
2. with bit-flip preprocessing.

Bit-flip preprocessing is an equivalent transformation up to a change of variables, so the
returned solution has the same objective value in the original QUBO space.

In [ ]:
solution_without_preprocessing = solvers.brute_force(instance)

print_solution("Solution without preprocessing:", solution_without_preprocessing)

Solution without preprocessing:
  labels bitstrings  costs  counts  probs  Trivial
0      0       1011   -2.0       1    1.0    False


In [ ]:
flipped_instance = transforms.negative_bitflip.apply(instance, time_limit_s=60.0)
flipped_solution = solvers.brute_force(flipped_instance)
solution_with_bitflip = transforms.negative_bitflip.unapply(flipped_solution, flipped_instance)

print_solution("Solution with bit-flip preprocessing:", solution_with_bitflip)

Solution with bit-flip preprocessing:
  labels bitstrings  costs  counts  probs  Trivial
0      0       1011   -2.0       1    1.0    False


In [ ]:
same_best_cost = math.isclose(solution_without_preprocessing[0].cost, solution_with_bitflip[0].cost)
print(f"\nSame best cost: {same_best_cost}")
assert same_best_cost


Same best cost: True


## 4. When bit-flip preprocessing is not enough

Bit-flip preprocessing does not always remove all negative coefficients.

In the next example, bit-flip preprocessing reduces the negative weight, but one negative coefficient remains.

In [ ]:
hard_instance = Instance(
    matrix.tensor(
    [
        [0.0, -2.0, 1.0, 1.0],
        [-2.0, 0.0, -2.0, 1.0],
        [1.0, -2.0, 0.0, -2.0],
        [1.0, 1.0, -2.0, 0.0],
    ])
)

print(f"QUBO:\n{hard_instance.matrix}")

QUBO:
tensor([[ 0., -2.,  1.,  1.],
        [-2.,  0., -2.,  1.],
        [ 1., -2.,  0., -2.],
        [ 1.,  1., -2.,  0.]])


In [ ]:
brute_force_solution = solvers.brute_force(hard_instance, max_bitstrings=10)
print_solution("Brute force solution:", brute_force_solution)

Brute force solution:
  labels bitstrings  costs  counts  probs  Trivial
0      0       1111   -6.0       1    0.1     True
1      0       1110   -6.0       1    0.1    False
2      0       0111   -6.0       1    0.1    False
3      0       0011   -4.0       1    0.1    False
4      0       1100   -4.0       1    0.1    False
5      0       0110   -4.0       1    0.1    False
6      0       1000    0.0       1    0.1    False
7      0       1101    0.0       1    0.1    False
8      0       1011    0.0       1    0.1    False
9      0       0000    0.0       1    0.1     True


In [ ]:
flipped_hard_instance = transforms.negative_bitflip.apply(hard_instance, time_limit_s=60.0)

print(f"Flip vector: {flipped_hard_instance.flips}")
print(f"Bitflip status: {flipped_hard_instance.status}")
print(f"Bitflip metrics: {json.dumps(flipped_hard_instance.metrics, indent=4)}")

print(f"\nPreprocessed QUBO without zeroing:\n{flipped_hard_instance.matrix}")

Flip vector: tensor([1, 0, 1, 0], dtype=torch.int8)
Bitflip status: OPTIMAL
Bitflip metrics: {
    "n_edges": 6,
    "neg_count_before": 3,
    "neg_count_after": 1,
    "neg_count_reduction_pct": 66.66666666666667,
    "neg_weight_before": 6.0,
    "neg_weight_after": 1.0,
    "neg_weight_reduction_pct": 83.33333333333333,
    "objective_value": -5.0
}

Preprocessed QUBO without zeroing:
tensor([[-2.,  2.,  1., -1.],
        [ 2., -8.,  2.,  1.],
        [ 1.,  2., -2.,  2.],
        [-1.,  1.,  2., -2.]])


## 5. Explicit zeroing of remaining negative coefficients

If negative off-diagonal coefficients remain after bit-flip preprocessing, they can be explicitly zeroed out with `transforms.zeroing.apply`, applied on top of the bit-flip result.

This makes the QUBO compatible with the quantum backend, but it changes the QUBO objective. It should therefore be used only when this approximation is acceptable.

In [ ]:
device = qoolqit.AnalogDeviceWithDMM()
emulator = LocalEmulator()

# Demonstrate the embedder's guard: embedding a QUBO that still has negative
# off-diagonal coefficients raises, instead of silently producing a wrong register.
try:
    register = embedding.greedy.embed(flipped_hard_instance, device)
except ValueError as e:
    print(f"\nExpected error, embedding rejects negative off-diagonal coefficients: {e}")

zeroed_hard_instance = transforms.zeroing.apply(flipped_hard_instance)
print(f"\nPreprocessed QUBO with zeroing:\n{zeroed_hard_instance.matrix}")

register = embedding.greedy.embed(zeroed_hard_instance, device)
drive = drive_shaping.heuristic.build_drive(zeroed_hard_instance, register, device=device, dmm=True)
job = solvers.analog_quantum_sample(register, drive, emulator, device)
zeroed_hard_quantum_solution = Solution.from_results(job.results(), instance=zeroed_hard_instance)
print_solution("\nQuantum Solution with zeroing:", zeroed_hard_quantum_solution)


Expected error, embedding rejects negative off-diagonal coefficients: QUBOs with negative off-diagonal coefficients cannot be embedded.

Preprocessed QUBO with zeroing:
tensor([[-2.,  2.,  1.,  0.],
        [ 2., -8.,  2.,  1.],
        [ 1.,  2., -2.,  2.],
        [ 0.,  1.,  2., -2.]])

Quantum Solution with zeroing:
  labels bitstrings  costs  counts  probs  Trivial
0      0       0101   -8.0     942  0.942    False
1      0       0100   -8.0      21  0.021    False
2      0       0110   -6.0      11  0.011    False
3      0       1101   -6.0       5  0.005    False
4      0       1100   -6.0       1  0.001    False
5      0       1001   -4.0       2  0.002    False
6      0       1010   -2.0      14  0.014    False
7      0       1110   -2.0       4  0.004    False


/home/bussy/work/qubo-solver.git/negative-coefficients-bitflip-preprocessing/qubosolver/drive_shaping/heuristic.py:119: DeprecationWarning: Interpolated is deprecated and will be removed in v1.4. Use the equivalent InterpolatedWaveform instead.
  amp_wave = qoolqit.Interpolated(
/home/bussy/work/qubo-solver.git/negative-coefficients-bitflip-preprocessing/qubosolver/drive_shaping/heuristic.py:125: DeprecationWarning: Interpolated is deprecated and will be removed in v1.4. Use the equivalent InterpolatedWaveform instead.
  det_wave = qoolqit.Interpolated(


The solution obtained is defined over the zeroed, flipped QUBO's variables. To recover a solution over the *original* QUBO's variables, both transforms must be undone — in the reverse order they were applied: first `zeroing.unapply`, then `negative_bitflip.unapply`.

In [ ]:
flipped_hard_quantum_solution = transforms.zeroing.unapply(zeroed_hard_quantum_solution, zeroed_hard_instance)
hard_quantum_solution = transforms.negative_bitflip.unapply(flipped_hard_quantum_solution, flipped_hard_instance)

print_solution("\nFinal Quantum Solution:", hard_quantum_solution)


Final Quantum Solution:
  labels bitstrings  costs  counts  probs  Trivial
0      0       1111   -6.0     942  0.942     True
1      0       1110   -6.0      21  0.021    False
2      0       1100   -4.0      11  0.011    False
3      0       0111   -6.0       5  0.005    False
4      0       0110   -4.0       1  0.001    False
5      0       0011   -4.0       2  0.002    False
6      0       0000    0.0      14  0.014     True
7      0       0100    0.0       4  0.004    False


## 6. Quantum solver behavior

In the `Solver` API, `config.do_preprocessing` controls whether bit-flip preprocessing runs before solving.

If negative off-diagonal coefficients remain after preprocessing, the quantum solver zeros them out automatically (and logs a message) before embedding, since the backend cannot encode negative interactions. It will also log a warning if negative off-diagonal coefficients are detected and preprocessing is not enabled.

In [ ]:
import logging

logging.basicConfig(level=logging.INFO)

config_quantum_no_preprocessing = SolverConfig(
    use_quantum=True,
    do_preprocessing=False,
    activate_trivial_solutions=False,
    do_postprocessing=False,
)

# Preprocessing is disabled: constructing the solver only warns, it does not raise.
quantum_solver = Solver(hard_instance, config_quantum_no_preprocessing)

Now we enable preprocessing.

Bit-flip preprocessing reduces the negative coefficients first. Any coefficient it cannot remove is zeroed automatically before the resulting QUBO is passed to the quantum backend.

In [ ]:
config_quantum = SolverConfig(
    use_quantum=True,
    do_preprocessing=True,
    activate_trivial_solutions=False,
    do_postprocessing=False,
)

quantum_solution = Solver(hard_instance, config_quantum).solve()

print_solution("Quantum solution bitstrings:", quantum_solution)

INFO:qubosolver.solvers.solver:Bit-flip preprocessing could not remove all negative off-diagonal coefficients; zeroing the remainder as an approximation before embedding.


Quantum solution bitstrings:
  labels bitstrings  costs  counts  probs  Trivial
0      0       1111   -6.0     956  0.956     True
1      0       1110   -6.0      18  0.018    False
2      0       0111   -6.0       3  0.003    False
3      0       0011   -4.0       4  0.004    False
4      0       1100   -4.0       8  0.008    False
5      0       0110   -4.0       2  0.002    False
6      0       0000    0.0       4  0.004     True
7      0       0100    0.0       5  0.005    False


/home/bussy/work/qubo-solver.git/negative-coefficients-bitflip-preprocessing/qubosolver/drive_shaping/heuristic.py:119: DeprecationWarning: Interpolated is deprecated and will be removed in v1.4. Use the equivalent InterpolatedWaveform instead.
  amp_wave = qoolqit.Interpolated(
/home/bussy/work/qubo-solver.git/negative-coefficients-bitflip-preprocessing/qubosolver/drive_shaping/heuristic.py:125: DeprecationWarning: Interpolated is deprecated and will be removed in v1.4. Use the equivalent InterpolatedWaveform instead.
  det_wave = qoolqit.Interpolated(


### Note on the quantum result

The quantum solution returned here is not necessarily the optimal solution of the original QUBO.

In this example, the quantum solver runs on a QUBO where the remaining negative off-diagonal coefficients have been set to zero. This makes the problem compatible with the quantum backend, but it also changes the objective optimized by the quantum dynamics.

The returned bitstrings are then evaluated on the original QUBO, which explains why the final costs should be interpreted as heuristic results.

Future work aims to go beyond this approximation by adding a dedicated method to encode negative interactions during the quantum resolution itself, in addition to the bit-flip preprocessing that already reduces them.

## Conclusion

Bit-flip preprocessing provides a practical way to reduce or remove negative off-diagonal QUBO coefficients before quantum solving.

It works by selecting a flip vector with GLPK. The flip vector indicates which binary variables should be complemented, and the transformed QUBO is equivalent to the original problem up to this variable change.

In this tutorial, GLPK is used to compute the flip vector exactly on small QUBOs. This is useful for validation and demonstration, but it is not expected to scale to large industrial instances. Future versions should replace this exact GLPK step with an efficient classical heuristic for selecting good flip vectors at larger scale.

When bit-flip preprocessing removes all negative off-diagonal coefficients, the QUBO can be passed directly to the quantum solver.

When some negative coefficients remain, the quantum solver falls back to `transforms.zeroing` automatically. This makes the QUBO compatible with the quantum backend, but it is an approximation because it changes the QUBO objective.

Future work should also add a dedicated method to represent negative interactions directly during the quantum resolution, going beyond the zeroing approximation used today.